# 8. Naive Bayes

**Machine Learning Fundamentals and Predictive Analytics — Notebook 8 of 11**

Naive Bayes applies Bayes' theorem (statistics Notebook 1) directly to classification, plus one
deliberately unrealistic assumption: that the features are **conditionally independent given the
class**.

That assumption is almost always false. Words in an email are not independent; "machine" makes
"learning" far more likely. And yet Naive Bayes works remarkably well — it is fast, it needs
almost no data, it handles thousands of features, and it was the original production spam
filter.

### What you will learn

1. From Bayes' theorem to a classifier
2. The **naive** conditional-independence assumption, and why it survives being wrong
3. **Gaussian**, **Multinomial**, **Bernoulli** and **Complement** variants
4. **Laplace smoothing** and the zero-probability problem
5. Why we work in **log space**
6. Building a spam classifier from scratch, then with scikit-learn
7. Why the probabilities are **poorly calibrated**
8. Strengths, weaknesses, and where Naive Bayes still wins

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.naive_bayes import (GaussianNB, MultinomialNB, BernoulliNB, ComplementNB)
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import (train_test_split, cross_val_score, GridSearchCV,
                                     StratifiedKFold, learning_curve)
from sklearn.preprocessing import StandardScaler, KBinsDiscretizer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (accuracy_score, confusion_matrix, classification_report,
                             roc_auc_score, log_loss, brier_score_loss,
                             precision_score, recall_score, f1_score,
                             average_precision_score, ConfusionMatrixDisplay)
from sklearn.calibration import calibration_curve, CalibratedClassifierCV
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
import time

rng = np.random.default_rng(seed=8)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 4)
SKF = StratifiedKFold(5, shuffle=True, random_state=0)

---
## 8.1 From Bayes' theorem to a classifier

We want $P(y = c \mid \mathbf{x})$ for each class $c$. Bayes' theorem gives

$$P(c \mid \mathbf{x}) = \frac{P(\mathbf{x} \mid c)\,P(c)}{P(\mathbf{x})}$$

Since $P(\mathbf{x})$ is the same for every class, we can drop it and just compare numerators:

$$\hat{y} = \arg\max_{c}\ P(c)\,P(\mathbf{x}\mid c)$$

The problem is $P(\mathbf{x}\mid c) = P(x_1, x_2, \dots, x_p \mid c)$ — a joint distribution
over all $p$ features. With 20 binary features that is $2^{20}$ combinations per class, and you
would need astronomical amounts of data to estimate it.

### The naive assumption

Assume the features are **conditionally independent given the class**:

$$P(x_1,\dots,x_p \mid c) = \prod_{j=1}^{p} P(x_j \mid c)$$

Now you only need $p$ one-dimensional distributions per class. The classifier becomes

$$\hat{y} = \arg\max_{c}\ P(c)\prod_{j=1}^{p}P(x_j \mid c)$$

and training is just counting. There is **no iterative optimisation** — one pass over the data
and you are done.

In [ ]:
# Naive Bayes by hand on a tiny categorical dataset
weather = pd.DataFrame({
    "outlook":  ["sunny","sunny","overcast","rain","rain","rain","overcast","sunny",
                 "sunny","rain","sunny","overcast","overcast","rain"],
    "temp":     ["hot","hot","hot","mild","cool","cool","cool","mild",
                 "cool","mild","mild","mild","hot","mild"],
    "humidity": ["high","high","high","high","normal","normal","normal","high",
                 "normal","normal","normal","high","normal","high"],
    "windy":    [False,True,False,False,False,True,True,False,
                 False,False,True,True,False,True],
    "play":     ["no","no","yes","yes","yes","no","yes","no",
                 "yes","yes","yes","yes","yes","no"],
})
print(weather.to_string(index=False))

# Priors
priors = weather.play.value_counts(normalize=True)
print(f"\nPriors: P(yes) = {priors['yes']:.4f}, P(no) = {priors['no']:.4f}")

In [ ]:
# Conditional probabilities: P(feature = value | class), one table per feature
features = ["outlook", "temp", "humidity", "windy"]
tables = {}
for f in features:
    ct = pd.crosstab(weather[f], weather.play, normalize="columns")
    tables[f] = ct
    print(f"\nP({f} | play):")
    print(ct.round(4).to_string())

In [ ]:
def naive_bayes_predict(sample, tables, priors, verbose=True):
    '''Compute the unnormalised posterior for each class, then normalise.'''
    posteriors = {}
    for c in priors.index:
        p = priors[c]
        parts = [f"P({c})={priors[c]:.4f}"]
        for f, v in sample.items():
            pv = tables[f].loc[v, c] if v in tables[f].index else 0.0
            p *= pv
            parts.append(f"P({f}={v}|{c})={pv:.4f}")
        posteriors[c] = p
        if verbose:
            print(f"  {c}: {' x '.join(parts)} = {p:.6f}")
    total = sum(posteriors.values())
    return {c: (v / total if total > 0 else 0.0) for c, v in posteriors.items()}

test_day = {"outlook": "sunny", "temp": "cool", "humidity": "high", "windy": True}
print(f"New day: {test_day}\n")
post = naive_bayes_predict(test_day, tables, priors)
print(f"\nNormalised posteriors: "
      f"{ {c: round(v, 4) for c, v in post.items()} }")
print(f"PREDICTION: play = {max(post, key=post.get)}")

---
## 8.2 The zero-probability problem, and Laplace smoothing

If a feature value never appears with a class in the training data, its conditional probability
is 0 — and because we **multiply** the probabilities, one zero wipes out the entire class. A
single unseen word would veto everything else the model knows.

**Laplace (add-$\alpha$) smoothing** fixes it by pretending we saw every combination
$\alpha$ extra times:

$$P(x_j = v \mid c) = \frac{\text{count}(x_j = v, c) + \alpha}{\text{count}(c) + \alpha \cdot |V_j|}$$

where $|V_j|$ is the number of possible values of feature $j$. $\alpha = 1$ is "Laplace
smoothing"; $\alpha < 1$ is "Lidstone smoothing". In scikit-learn the parameter is `alpha`, and
it is worth tuning — for text it is one of the few knobs that matters.

In [ ]:
# A class that never co-occurs with a value: 'overcast' always means play=yes
print("Counts of outlook by class:")
print(pd.crosstab(weather.outlook, weather.play).to_string())
print("\nNote: outlook='overcast' NEVER appears with play='no'.")
print(f"So P(overcast | no) = {tables['outlook'].loc['overcast', 'no']:.4f}\n")

overcast_day = {"outlook": "overcast", "temp": "hot", "humidity": "high", "windy": True}
print(f"Unsmoothed prediction for {overcast_day}:")
post_raw = naive_bayes_predict(overcast_day, tables, priors)
print(f"  -> {{'no': {post_raw['no']:.6f}, 'yes': {post_raw['yes']:.6f}}}")
print("  The 'no' class is IMPOSSIBLE, no matter how strongly the other features argue.\n")

def smoothed_tables(frame, features, target, alpha=1.0):
    out = {}
    for f in features:
        counts = pd.crosstab(frame[f], frame[target])
        n_vals = frame[f].nunique()
        out[f] = (counts + alpha) / (counts.sum(axis=0) + alpha * n_vals)
    return out

for a in (0.0001, 1.0):
    st = smoothed_tables(weather, features, "play", alpha=a)
    p = naive_bayes_predict(overcast_day, st, priors, verbose=False)
    print(f"alpha = {a:<8} -> P(no) = {p['no']:.6f}, P(yes) = {p['yes']:.6f}")
print("\nWith alpha = 1 the 'no' class is unlikely but not impossible -- which is the")
print("honest position when you have only 14 training examples.")

---
## 8.3 Working in log space

Multiplying thousands of probabilities underflows to zero in floating point. Since $\log$ is
monotone, we take logs and add instead:

$$\log P(c \mid \mathbf{x}) \propto \log P(c) + \sum_{j=1}^{p}\log P(x_j \mid c)$$

Every real implementation does this. `predict_log_proba` exposes it.

In [ ]:
n_features_demo = 2_000
p_typical = 0.01
print(f"Multiplying {n_features_demo:,} probabilities of about {p_typical}:")
print(f"  direct product : {p_typical ** n_features_demo}   <- underflowed to zero")
print(f"  in log space   : {n_features_demo * np.log(p_typical):.1f}   <- perfectly fine")
print("\nWhere the smallest representable positive float is about "
      f"{np.finfo(float).tiny:.1e}, so a product of about "
      f"{int(np.log(np.finfo(float).tiny)/np.log(p_typical))} such terms is the limit.")
print("\nTo recover normalised probabilities from log-scores, subtract the max before")
print("exponentiating (the 'log-sum-exp trick'):")
log_scores = np.array([-3200.5, -3198.1, -3210.7])
shifted = log_scores - log_scores.max()
probs = np.exp(shifted) / np.exp(shifted).sum()
print(f"  log scores {log_scores} -> probabilities {probs.round(4)}")

---
## 8.4 The four variants

The variants differ only in how they model $P(x_j \mid c)$.

| Variant | Assumes each feature is | Use for |
|---|---|---|
| **GaussianNB** | Normally distributed within each class | Continuous features |
| **MultinomialNB** | A count from a multinomial distribution | Word **counts** or TF-IDF; the text default |
| **BernoulliNB** | A binary indicator | Binary features, or "word present/absent" |
| **ComplementNB** | Multinomial, but parameters estimated from the *complement* of each class | **Imbalanced** text data |

Getting this choice wrong is the most common Naive Bayes mistake: `GaussianNB` on word counts,
or `MultinomialNB` on standardised (negative) features, which raises an error.

In [ ]:
# GaussianNB: it fits one mean and variance per feature per class
X_g, y_g = (lambda: (np.vstack([rng.normal([2, 2], [1.0, 0.6], (200, 2)),
                                rng.normal([5, 4], [0.8, 1.4], (200, 2))]),
                     np.r_[np.zeros(200), np.ones(200)].astype(int)))()

gnb = GaussianNB().fit(X_g, y_g)
print("Learned parameters (one per class per feature):")
print(pd.DataFrame({"class": [0, 0, 1, 1], "feature": [0, 1, 0, 1],
                    "mean": gnb.theta_.ravel(),
                    "variance": gnb.var_.ravel()}).round(4).to_string(index=False))
print(f"\nClass priors: {gnb.class_prior_.round(4)}")
print("\nThat is the entire model: 2 classes x 2 features x 2 parameters, plus priors.")

xx, yy = np.meshgrid(np.linspace(-1, 8, 300), np.linspace(-1, 8, 300))
Z = gnb.predict_proba(np.c_[xx.ravel(), yy.ravel()])[:, 1].reshape(xx.shape)
fig, ax = plt.subplots(1, 2, figsize=(13, 4.6))
c0 = ax[0].contourf(xx, yy, Z, levels=20, cmap="coolwarm", alpha=0.7)
plt.colorbar(c0, ax=ax[0], label="P(class 1)")
ax[0].scatter(X_g[:, 0], X_g[:, 1], c=y_g, cmap="coolwarm", s=12, edgecolor="k", linewidth=0.2)
ax[0].contour(xx, yy, Z, levels=[0.5], colors="k", linewidths=2)
ax[0].set_title("GaussianNB: quadratic boundary from axis-aligned Gaussians")

# Show the fitted per-class densities for feature 0
xs = np.linspace(-1, 8, 400)
from scipy.stats import norm
for c, colour in [(0, "steelblue"), (1, "crimson")]:
    ax[1].plot(xs, norm(gnb.theta_[c, 0], np.sqrt(gnb.var_[c, 0])).pdf(xs),
               color=colour, lw=2, label=f"P(x0 | class {c})")
    ax[1].hist(X_g[y_g == c, 0], bins=25, density=True, alpha=0.3, color=colour)
ax[1].set_title("The 1-D densities the model actually fits"); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# What the naive assumption costs when features are correlated
def make_correlated(rho, m=600, seed=0):
    g = np.random.default_rng(seed)
    cov = np.array([[1.0, rho], [rho, 1.0]])
    Xa_ = g.multivariate_normal([0, 0], cov, m//2)
    Xb_ = g.multivariate_normal([2, 2], cov, m//2)
    return np.vstack([Xa_, Xb_]), np.r_[np.zeros(m//2), np.ones(m//2)].astype(int)

print(f"{'feature correlation':>21}{'GaussianNB':>13}{'logistic':>11}{'gap':>8}")
for rho in (0.0, 0.3, 0.6, 0.85, 0.95):
    Xr_, yr_ = make_correlated(rho)
    nb_acc = cross_val_score(GaussianNB(), Xr_, yr_, cv=SKF).mean()
    lr_acc = cross_val_score(make_pipeline(StandardScaler(), LogisticRegression()),
                             Xr_, yr_, cv=SKF).mean()
    print(f"{rho:>21.2f}{nb_acc:>13.4f}{lr_acc:>11.4f}{lr_acc-nb_acc:>8.4f}")
print("\nWith independent features Naive Bayes matches logistic regression. As correlation")
print("grows, its assumption is violated and it falls behind -- but note how SLOWLY.")
print("Even at rho = 0.95 the gap is small, which is the puzzle the next section explains.")

### Why does it work when the assumption is false?

Two reasons:

1. **Classification only needs the argmax.** Naive Bayes can get the probabilities badly wrong
   and still rank the classes correctly. Double-counting correlated evidence pushes the
   probability toward 0 or 1, but usually *in the right direction*.
2. **Few parameters means low variance.** With $p$ features you estimate $O(p)$ parameters
   instead of $O(2^p)$. The assumption injects bias, and in the bias-variance trade-off
   (statistics Notebook 12) that bias buys a large variance reduction — which is exactly the
   right trade when data is scarce.

The corollary: Naive Bayes is a **good classifier and a bad probability estimator**.

In [ ]:
# The direction is right even when the magnitude is not
Xd_, yd_ = make_correlated(0.9, m=2000, seed=1)
Xa_, Xb_, ya_, yb_ = train_test_split(Xd_, yd_, test_size=0.3, random_state=0, stratify=yd_)
nb = GaussianNB().fit(Xa_, ya_)
lr = make_pipeline(StandardScaler(), LogisticRegression()).fit(Xa_, ya_)

pb_nb, pb_lr = nb.predict_proba(Xb_)[:, 1], lr.predict_proba(Xb_)[:, 1]
print(f"Accuracy    : NB {accuracy_score(yb_, nb.predict(Xb_)):.4f}   "
      f"LR {accuracy_score(yb_, lr.predict(Xb_)):.4f}")
print(f"ROC-AUC     : NB {roc_auc_score(yb_, pb_nb):.4f}   LR {roc_auc_score(yb_, pb_lr):.4f}")
print(f"Log loss    : NB {log_loss(yb_, pb_nb):.4f}   LR {log_loss(yb_, pb_lr):.4f}  <- much worse")
print(f"Brier score : NB {brier_score_loss(yb_, pb_nb):.4f}   "
      f"LR {brier_score_loss(yb_, pb_lr):.4f}")

plt.hist(pb_nb, bins=40, alpha=0.6, color="crimson", label="GaussianNB")
plt.hist(pb_lr, bins=40, alpha=0.6, color="steelblue", label="logistic regression")
plt.xlabel("predicted P(class 1)"); plt.ylabel("count")
plt.title("Naive Bayes pushes probabilities to the extremes")
plt.legend(fontsize=8); plt.show()
print(f"\nFraction of NB predictions above 0.99 or below 0.01: "
      f"{((pb_nb > 0.99) | (pb_nb < 0.01)).mean():.3f}")
print(f"Same for logistic regression: {((pb_lr > 0.99) | (pb_lr < 0.01)).mean():.3f}")
print("\nAccuracy and AUC are nearly identical; the probabilities are not. If you only")
print("need a decision, use NB freely. If you need the number, calibrate it or use LR.")

---
## 8.5 Text classification: the natural home

Naive Bayes and text are made for each other:

- Thousands of features (one per word) — the $O(p)$ parameter count keeps this tractable
- Sparse counts — multinomial likelihoods handle them naturally
- Training is one pass of counting, so it scales to enormous corpora
- It works with very little labelled data

We build a spam classifier by hand first, then with scikit-learn.

In [ ]:
# A small labelled corpus
spam_msgs = [
    "win a free iphone click here now",
    "congratulations you won a free prize claim now",
    "free money click this link now",
    "urgent your account will be closed click here",
    "you have won a lottery claim your prize money",
    "cheap meds free shipping order now",
    "click here to claim free bitcoin now",
    "limited offer win cash prize free entry",
    "your loan is approved click to claim money",
    "free vacation offer click now urgent",
]
ham_msgs = [
    "can we move the meeting to three pm",
    "please review the attached report before friday",
    "the project deadline has been extended by a week",
    "lunch tomorrow at the usual place",
    "here are the meeting notes from monday",
    "could you send me the sales figures",
    "the client approved the new design",
    "reminder about the team meeting on wednesday",
    "i will be working from home tomorrow",
    "thanks for sending the report yesterday",
]
docs = spam_msgs + ham_msgs
labels = np.r_[np.ones(len(spam_msgs)), np.zeros(len(ham_msgs))].astype(int)
print(f"{len(spam_msgs)} spam, {len(ham_msgs)} ham messages")

# Build the vocabulary and count words per class, by hand
def tokenise(text):
    return text.lower().split()

vocab = sorted({w for d in docs for w in tokenise(d)})
print(f"Vocabulary size: {len(vocab)}")

counts = {1: {}, 0: {}}
totals = {1: 0, 0: 0}
for d, lab in zip(docs, labels):
    for w in tokenise(d):
        counts[lab][w] = counts[lab].get(w, 0) + 1
        totals[lab] += 1
print(f"Total word occurrences: spam {totals[1]}, ham {totals[0]}")

In [ ]:
ALPHA = 1.0
V = len(vocab)

def word_logprob(word, cls):
    '''Laplace-smoothed log P(word | class).'''
    return np.log((counts[cls].get(word, 0) + ALPHA) / (totals[cls] + ALPHA * V))

log_prior = {c: np.log((labels == c).sum() / len(labels)) for c in (0, 1)}

def classify(text, explain=False):
    scores = {}
    for c in (0, 1):
        s = log_prior[c]
        contribs = []
        for w in tokenise(text):
            lp = word_logprob(w, c)
            s += lp
            contribs.append((w, lp))
        scores[c] = s
        if explain:
            print(f"  class {'spam' if c else 'ham'}: log-prior {log_prior[c]:.3f} "
                  f"+ " + " + ".join(f"{w}:{lp:.2f}" for w, lp in contribs) + f" = {s:.3f}")
    mx = max(scores.values())
    exps = {c: np.exp(v - mx) for c, v in scores.items()}
    tot = sum(exps.values())
    return {c: exps[c]/tot for c in exps}, max(scores, key=scores.get)

for msg in ["free money click now", "can we schedule the report meeting",
            "urgent claim your free prize"]:
    print(f"\n'{msg}'")
    probs, pred = classify(msg, explain=True)
    print(f"  -> P(spam) = {probs[1]:.4f}  PREDICTION: {'SPAM' if pred else 'HAM'}")

In [ ]:
# Which words carry the most weight? The log-likelihood ratio.
ratios = []
for w in vocab:
    ratios.append({"word": w,
                   "log_ratio": word_logprob(w, 1) - word_logprob(w, 0),
                   "spam_count": counts[1].get(w, 0),
                   "ham_count": counts[0].get(w, 0)})
R = pd.DataFrame(ratios).sort_values("log_ratio", ascending=False)
print("Most spam-indicative words:")
print(R.head(8).round(3).to_string(index=False))
print("\nMost ham-indicative words:")
print(R.tail(8).round(3).to_string(index=False))
print("\nThis table IS the model, and it is completely auditable -- one of the underrated")
print("virtues of Naive Bayes for text.")

In [ ]:
# The same thing with scikit-learn, on a larger synthetic corpus
spam_words = ["free", "win", "click", "now", "urgent", "prize", "claim", "money",
              "offer", "cheap", "limited", "cash", "bitcoin", "loan", "guaranteed"]
ham_words = ["meeting", "report", "project", "deadline", "review", "team", "client",
             "schedule", "attached", "notes", "figures", "design", "tomorrow", "please",
             "thanks"]
filler = ["the", "a", "to", "and", "of", "for", "your", "you", "we", "is", "on", "in"]

def make_message(is_spam, g):
    topic = spam_words if is_spam else ham_words
    n_topic = g.integers(2, 6)
    n_filler = g.integers(3, 8)
    n_cross = g.integers(0, 2)                    # a few words from the other class
    other = ham_words if is_spam else spam_words
    words = (list(g.choice(topic, n_topic)) + list(g.choice(filler, n_filler))
             + list(g.choice(other, n_cross)))
    g.shuffle(words)
    return " ".join(words)

m_text = 2_000
y_text = (rng.random(m_text) < 0.35).astype(int)
X_text = [make_message(bool(t), rng) for t in y_text]
print(f"{m_text} messages, spam rate {y_text.mean():.3f}\n")
for i in range(3):
    print(f"  [{'SPAM' if y_text[i] else 'HAM '}] {X_text[i]}")

Xt_tr, Xt_te, yt_tr, yt_te = train_test_split(X_text, y_text, test_size=0.3,
                                              random_state=0, stratify=y_text)

In [ ]:
# Compare the vectoriser / variant combinations
combos = {
    "CountVectorizer + MultinomialNB": make_pipeline(CountVectorizer(), MultinomialNB()),
    "TfidfVectorizer + MultinomialNB": make_pipeline(TfidfVectorizer(), MultinomialNB()),
    "Binary counts + BernoulliNB": make_pipeline(CountVectorizer(binary=True), BernoulliNB()),
    "CountVectorizer + ComplementNB": make_pipeline(CountVectorizer(), ComplementNB()),
    "TfidfVectorizer + LogisticRegression": make_pipeline(TfidfVectorizer(),
                                                          LogisticRegression(max_iter=2000)),
}
print(f"{'pipeline':<40}{'CV acc':>9}{'test acc':>10}{'test AUC':>10}{'fit s':>8}")
for name, est in combos.items():
    t0 = time.perf_counter()
    cvv = cross_val_score(est, Xt_tr, yt_tr, cv=SKF).mean()
    fit_s = (time.perf_counter() - t0) / 5
    est.fit(Xt_tr, yt_tr)
    pb = est.predict_proba(Xt_te)[:, 1]
    print(f"{name:<40}{cvv:>9.4f}{est.score(Xt_te, yt_te):>10.4f}"
          f"{roc_auc_score(yt_te, pb):>10.4f}{fit_s:>8.3f}")
print(f"\nBaseline (majority class): "
      f"{DummyClassifier(strategy='most_frequent').fit(Xt_tr, yt_tr).score(Xt_te, yt_te):.4f}")
print("\nNaive Bayes matches logistic regression here and fits in a fraction of the time.")
print("On text, that speed advantage is the whole argument.")

In [ ]:
best_text = make_pipeline(CountVectorizer(), MultinomialNB(alpha=0.5)).fit(Xt_tr, yt_tr)
pred_t = best_text.predict(Xt_te)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ConfusionMatrixDisplay(confusion_matrix(yt_te, pred_t),
                       display_labels=["ham", "spam"]).plot(ax=ax[0], cmap="Blues",
                                                            colorbar=False)
ax[0].set_title("Confusion matrix")

vec = best_text.named_steps["countvectorizer"]
nbm = best_text.named_steps["multinomialnb"]
log_ratio = nbm.feature_log_prob_[1] - nbm.feature_log_prob_[0]
words = np.array(vec.get_feature_names_out())
order = np.argsort(log_ratio)
top = np.r_[order[:10], order[-10:]]
colors = ["steelblue"] * 10 + ["crimson"] * 10
ax[1].barh(words[top], log_ratio[top], color=colors)
ax[1].axvline(0, color="black", lw=1)
ax[1].set_xlabel("log P(word|spam) - log P(word|ham)")
ax[1].set_title("Most discriminative words")
plt.tight_layout(); plt.show()

print(classification_report(yt_te, pred_t, target_names=["ham", "spam"]))

In [ ]:
# Tuning alpha, and why it matters more than you would expect
alphas = np.logspace(-3, 2, 25)
scores = [cross_val_score(make_pipeline(CountVectorizer(), MultinomialNB(alpha=a)),
                          Xt_tr, yt_tr, cv=SKF).mean() for a in alphas]
plt.plot(alphas, scores, "o-", color="steelblue")
best_a = alphas[int(np.argmax(scores))]
plt.axvline(best_a, color="crimson", ls="--", label=f"best alpha = {best_a:.4f}")
plt.axvline(1.0, color="grey", ls=":", label="default alpha = 1.0")
plt.xscale("log"); plt.xlabel("alpha (smoothing)"); plt.ylabel("CV accuracy")
plt.title("Smoothing is the main Naive Bayes hyperparameter")
plt.legend(fontsize=8); plt.show()

print(f"Best CV accuracy {max(scores):.4f} at alpha = {best_a:.4f}")
print(f"Accuracy at the default alpha=1.0: "
      f"{scores[int(np.argmin(np.abs(alphas - 1.0)))]:.4f}")
print("\nToo little smoothing: rare words dominate and the model overfits.")
print("Too much: every conditional probability flattens toward uniform and the model")
print("degenerates to predicting the prior.")

In [ ]:
# Naive Bayes needs remarkably little data -- its headline advantage
fig, ax = plt.subplots(figsize=(8, 4.4))
for name, est, colour in [
        ("MultinomialNB", make_pipeline(CountVectorizer(), MultinomialNB()), "crimson"),
        ("logistic regression", make_pipeline(TfidfVectorizer(),
                                              LogisticRegression(max_iter=2000)), "steelblue"),
        ("random forest", make_pipeline(CountVectorizer(),
                                        RandomForestClassifier(n_estimators=100,
                                                               random_state=0)), "seagreen")]:
    sizes, tr, va = learning_curve(est, Xt_tr, yt_tr,
                                   train_sizes=[0.02, 0.05, 0.1, 0.2, 0.4, 0.7, 1.0],
                                   cv=SKF, scoring="accuracy")
    ax.plot(sizes, va.mean(1), "o-", color=colour, label=name)
ax.set_xlabel("training messages"); ax.set_ylabel("cross-validated accuracy")
ax.set_title("Naive Bayes reaches its ceiling with very few examples")
ax.legend(fontsize=8); plt.show()

print("With only ~30 labelled messages Naive Bayes is already near its best. This is why")
print("it remains the go-to for bootstrapping a classifier before you have labelled data")
print("at scale, and why it is a standard baseline in every text-classification paper.")

---
## 8.6 Calibration: the known weakness

Because it double-counts correlated evidence, Naive Bayes produces probabilities that are
systematically pushed toward 0 and 1. It is **over-confident**.

That does not hurt accuracy or AUC (the ranking is preserved), but it wrecks anything that uses
the probability as a number: expected-value calculations, cost-based thresholds, risk scores.

The fix is `CalibratedClassifierCV` with `method="isotonic"` (flexible, needs a few hundred
samples) or `"sigmoid"` (Platt scaling, works with less data).

In [ ]:
nb_raw = make_pipeline(CountVectorizer(), MultinomialNB()).fit(Xt_tr, yt_tr)
nb_iso = CalibratedClassifierCV(make_pipeline(CountVectorizer(), MultinomialNB()),
                                method="isotonic", cv=5).fit(Xt_tr, yt_tr)
nb_sig = CalibratedClassifierCV(make_pipeline(CountVectorizer(), MultinomialNB()),
                                method="sigmoid", cv=5).fit(Xt_tr, yt_tr)
lr_text = make_pipeline(TfidfVectorizer(), LogisticRegression(max_iter=2000)).fit(Xt_tr, yt_tr)

plt.plot([0, 1], [0, 1], "k--", lw=1.4, label="perfect calibration")
for name, mdl in [("MultinomialNB (raw)", nb_raw),
                  ("NB + isotonic", nb_iso),
                  ("NB + sigmoid", nb_sig),
                  ("logistic regression", lr_text)]:
    pb = mdl.predict_proba(Xt_te)[:, 1]
    fp, mp = calibration_curve(yt_te, pb, n_bins=10, strategy="quantile")
    plt.plot(mp, fp, "o-", lw=2,
             label=f"{name} (Brier {brier_score_loss(yt_te, pb):.4f})")
plt.xlabel("mean predicted probability"); plt.ylabel("observed fraction positive")
plt.title("Naive Bayes needs calibration; calibration fixes it")
plt.legend(fontsize=8); plt.show()

print(f"{'model':<26}{'accuracy':>10}{'ROC_AUC':>10}{'log loss':>11}{'Brier':>9}")
for name, mdl in [("NB raw", nb_raw), ("NB + isotonic", nb_iso),
                  ("NB + sigmoid", nb_sig), ("logistic regression", lr_text)]:
    pb = mdl.predict_proba(Xt_te)[:, 1]
    print(f"{name:<26}{mdl.score(Xt_te, yt_te):>10.4f}{roc_auc_score(yt_te, pb):>10.4f}"
          f"{log_loss(yt_te, pb):>11.4f}{brier_score_loss(yt_te, pb):>9.4f}")
print("\nCalibration barely changes accuracy or AUC and transforms the log loss and Brier")
print("score. That is the signature of a ranking-good, probability-bad model.")

---
## 8.7 Strengths, weaknesses, and when to use it

**Strengths**

- **Extremely fast** — one counting pass to train, a dot product to predict
- **Tiny data requirement** — works with dozens of examples
- **Scales to huge feature counts** — text with 100,000 words is routine
- **Naturally multiclass** — no one-vs-rest machinery
- **Interpretable** — the log-likelihood-ratio table is the model
- **Streaming friendly** — `partial_fit` lets you update on new data
- No hyperparameter tuning beyond `alpha`

**Weaknesses**

- The **independence assumption** is false, and costs accuracy when features are strongly
  correlated
- **Badly calibrated** probabilities
- Cannot learn **interactions** at all
- `GaussianNB` imposes a Normal, axis-aligned shape that rarely fits real features
- The zero-frequency problem needs smoothing
- Usually beaten by logistic regression or gradient boosting given enough data

**Use it when:** text or high-dimensional sparse data, very little labelled data, a hard latency
or memory budget, streaming updates, or you need a fast honest baseline in one line.

In [ ]:
# Where each model wins: a head-to-head across three regimes
def compare(Xd, yd, label, text=False):
    if text:
        models = {
            "MultinomialNB": make_pipeline(CountVectorizer(), MultinomialNB()),
            "logistic regression": make_pipeline(TfidfVectorizer(),
                                                 LogisticRegression(max_iter=2000)),
            "random forest": make_pipeline(CountVectorizer(),
                                           RandomForestClassifier(n_estimators=200,
                                                                  random_state=0)),
        }
    else:
        models = {
            "GaussianNB": GaussianNB(),
            "logistic regression": make_pipeline(StandardScaler(),
                                                 LogisticRegression(max_iter=2000)),
            "random forest": RandomForestClassifier(n_estimators=200, random_state=0),
        }
    print(f"\n{label}")
    for name, est in models.items():
        cvv = cross_val_score(est, Xd, yd, cv=SKF, scoring="accuracy")
        print(f"  {name:<22} CV accuracy {cvv.mean():.4f} +/- {cvv.std():.4f}")

# (a) text, plenty of data
compare(Xt_tr, yt_tr, "(a) TEXT, 1400 messages:", text=True)

# (b) text, almost no data
compare(Xt_tr[:40], yt_tr[:40], "(b) TEXT, only 40 messages:", text=True)

# (c) tabular data with a strong interaction
m_i = 1_500
Xi = rng.normal(size=(m_i, 6))
yi = ((Xi[:, 0] * Xi[:, 1] > 0.3) | (Xi[:, 2] > 1.5)).astype(int)
compare(Xi, yi, "(c) TABULAR with an XOR-like interaction:")
print("\nNaive Bayes wins on small text, ties on large text, and is helpless against an")
print("interaction -- it literally cannot represent 'x0 AND x1' because it multiplies")
print("independent per-feature terms.")

---
## Exercises

**Exercise 1.** Work the weather example by hand for a new day, with and without Laplace
smoothing, then verify against `sklearn`'s `CategoricalNB`. Explain any difference.

In [ ]:
# --- Solution 1 -------------------------------------------------------------
from sklearn.naive_bayes import CategoricalNB
from sklearn.preprocessing import OrdinalEncoder

new_day = {"outlook": "rain", "temp": "hot", "humidity": "high", "windy": True}
print(f"New day: {new_day}\n")

print("Unsmoothed, by hand:")
p_raw = naive_bayes_predict(new_day, tables, priors)
print(f"  -> P(yes) = {p_raw['yes']:.6f}, P(no) = {p_raw['no']:.6f}\n")

print("With Laplace smoothing (alpha = 1), by hand:")
st = smoothed_tables(weather, features, "play", alpha=1.0)
p_sm = naive_bayes_predict(new_day, st, priors, verbose=True)
print(f"  -> P(yes) = {p_sm['yes']:.6f}, P(no) = {p_sm['no']:.6f}\n")

enc = OrdinalEncoder()
Xw_enc = enc.fit_transform(weather[features].astype(str))
yw_enc = (weather.play == "yes").astype(int)
cnb = CategoricalNB(alpha=1.0).fit(Xw_enc, yw_enc)
new_enc = enc.transform(pd.DataFrame([new_day])[features].astype(str))
sk_p = cnb.predict_proba(new_enc)[0]
print(f"sklearn CategoricalNB(alpha=1): P(no) = {sk_p[0]:.6f}, P(yes) = {sk_p[1]:.6f}")
print(f"Our smoothed calculation      : P(no) = {p_sm['no']:.6f}, P(yes) = {p_sm['yes']:.6f}")
print("\nThey match to several decimals. Any tiny difference comes from how the class")
print("priors are smoothed (sklearn also applies alpha to the class counts by default")
print("via fit_prior), which matters only for datasets this small.")

**Exercise 2.** Build a topic classifier for four categories using `MultinomialNB`. Report the
confusion matrix, identify which pair of topics is confused most, and show the top words for
each class.

In [ ]:
# --- Solution 2 -------------------------------------------------------------
topics = {
    "sport":    ["match", "goal", "team", "player", "coach", "league", "score", "tournament",
                 "injury", "season", "striker", "defence"],
    "finance":  ["market", "shares", "profit", "revenue", "investor", "bank", "interest",
                 "quarterly", "dividend", "loss", "trading", "portfolio"],
    "tech":     ["software", "server", "release", "api", "developer", "cloud", "database",
                 "latency", "deploy", "bug", "feature", "framework"],
    "politics": ["election", "minister", "parliament", "policy", "vote", "party", "campaign",
                 "bill", "coalition", "reform", "cabinet", "debate"],
}
common = ["the", "a", "to", "of", "and", "in", "for", "on", "said", "new", "after", "has"]

def make_article(topic, g, n_topic=6, n_common=8, n_leak=2):
    others = [w for t, ws in topics.items() if t != topic for w in ws]
    words = (list(g.choice(topics[topic], n_topic))
             + list(g.choice(common, n_common))
             + list(g.choice(others, n_leak)))
    g.shuffle(words)
    return " ".join(words)

names = list(topics)
m_t = 2_400
y_topic = rng.integers(0, 4, m_t)
X_topic = [make_article(names[t], rng) for t in y_topic]
Xa6, Xb6, ya6, yb6 = train_test_split(X_topic, y_topic, test_size=0.3, random_state=0,
                                      stratify=y_topic)

clf = make_pipeline(CountVectorizer(), MultinomialNB(alpha=0.3)).fit(Xa6, ya6)
pred6 = clf.predict(Xb6)
print(f"Test accuracy: {accuracy_score(yb6, pred6):.4f}")
print(f"Baseline     : "
      f"{DummyClassifier(strategy='most_frequent').fit(Xa6, ya6).score(Xb6, yb6):.4f}\n")
print(classification_report(yb6, pred6, target_names=names))

In [ ]:
cm6 = confusion_matrix(yb6, pred6)
fig, ax = plt.subplots(1, 2, figsize=(13, 4.6))
sns.heatmap(cm6, annot=True, fmt="d", cmap="Blues", xticklabels=names, yticklabels=names,
            ax=ax[0], cbar=False)
ax[0].set_xlabel("predicted"); ax[0].set_ylabel("true"); ax[0].set_title("Confusion matrix")

off = cm6.copy(); np.fill_diagonal(off, 0)
i, j = np.unravel_index(np.argmax(off), off.shape)
print(f"Most confused pair: true '{names[i]}' predicted as '{names[j]}' "
      f"({off[i, j]} times)")

vec6 = clf.named_steps["countvectorizer"]
nb6 = clf.named_steps["multinomialnb"]
words6 = np.array(vec6.get_feature_names_out())
rows = []
for c, nm in enumerate(names):
    # log P(word|class) minus the mean over the other classes
    others = np.delete(np.arange(4), c)
    score = nb6.feature_log_prob_[c] - nb6.feature_log_prob_[others].mean(axis=0)
    top = np.argsort(score)[-6:][::-1]
    rows.append({"topic": nm, "top words": ", ".join(words6[top])})
print()
print(pd.DataFrame(rows).to_string(index=False))

ax[1].barh(range(6), np.sort(nb6.feature_log_prob_[0] -
                             nb6.feature_log_prob_[[1, 2, 3]].mean(axis=0))[-6:],
           color="steelblue")
ax[1].set_yticks(range(6))
ax[1].set_yticklabels(words6[np.argsort(nb6.feature_log_prob_[0] -
                                        nb6.feature_log_prob_[[1,2,3]].mean(axis=0))[-6:]])
ax[1].set_title(f"Most distinctive words for '{names[0]}'")
plt.tight_layout(); plt.show()
print("\nThe recovered top words match the vocabularies used to generate each topic,")
print("and the confusions come from the deliberate 'leak' words shared across topics.")

**Exercise 3.** Show that Naive Bayes is over-confident and fix it. Use a dataset with strongly
correlated features, report log loss and Brier score before and after calibration, and explain
what a business decision would get wrong if you used the raw probabilities.

In [ ]:
# --- Solution 3 -------------------------------------------------------------
# 10 features, all highly correlated with a latent factor -> maximal double-counting
m_o = 4_000
latent = rng.normal(size=m_o)
cls = (latent > 0).astype(int)
Xo = np.column_stack([latent + rng.normal(0, 0.55, m_o) for _ in range(10)])
Xa7, Xb7, ya7, yb7 = train_test_split(Xo, cls, test_size=0.3, random_state=0, stratify=cls)
print(f"Mean pairwise feature correlation: "
      f"{np.corrcoef(Xo, rowvar=False)[np.triu_indices(10, 1)].mean():.3f}\n")

raw_nb = GaussianNB().fit(Xa7, ya7)
cal_nb = CalibratedClassifierCV(GaussianNB(), method="isotonic", cv=5).fit(Xa7, ya7)
lr7 = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(Xa7, ya7)

print(f"{'model':<26}{'accuracy':>10}{'ROC_AUC':>10}{'log loss':>11}{'Brier':>9}")
for name, mdl in [("GaussianNB (raw)", raw_nb), ("GaussianNB (calibrated)", cal_nb),
                  ("logistic regression", lr7)]:
    pb = mdl.predict_proba(Xb7)[:, 1]
    print(f"{name:<26}{mdl.score(Xb7, yb7):>10.4f}{roc_auc_score(yb7, pb):>10.4f}"
          f"{log_loss(yb7, pb):>11.4f}{brier_score_loss(yb7, pb):>9.4f}")

pb_raw = raw_nb.predict_proba(Xb7)[:, 1]
print(f"\nOf the raw NB predictions, {(pb_raw > 0.999).mean() + (pb_raw < 0.001).mean():.1%} "
      f"are above 0.999 or below 0.001.")

In [ ]:
# What a business decision gets wrong
print("Scenario: we act when P(event) > 0.9. Acting costs 100; a correct action gains 1000.\n")
for name, mdl in [("raw GaussianNB", raw_nb), ("calibrated GaussianNB", cal_nb),
                  ("logistic regression", lr7)]:
    pb = mdl.predict_proba(Xb7)[:, 1]
    act = pb > 0.9
    if act.sum() == 0:
        print(f"  {name:<24} acts on 0 cases")
        continue
    hit_rate = yb7[act].mean()
    profit = act.sum() * (hit_rate * 1000 - 100)
    print(f"  {name:<24} acts on {act.sum():>4} cases; the model CLAIMS >90% will pay off,")
    print(f"  {'':<24} actual hit rate {hit_rate:.3f}, realised profit {profit:>9,.0f}")

fp, mp = calibration_curve(yb7, pb_raw, n_bins=10, strategy="quantile")
plt.plot([0, 1], [0, 1], "k--", label="perfect")
plt.plot(mp, fp, "o-", color="crimson", lw=2, label="raw GaussianNB")
pb_cal = cal_nb.predict_proba(Xb7)[:, 1]
fp2, mp2 = calibration_curve(yb7, pb_cal, n_bins=10, strategy="quantile")
plt.plot(mp2, fp2, "o-", color="seagreen", lw=2, label="calibrated")
plt.xlabel("predicted probability"); plt.ylabel("observed frequency")
plt.title("Over-confidence, and its cure"); plt.legend(fontsize=8); plt.show()

print("\nWhat goes wrong with raw probabilities:")
print("  * 'P > 0.9' selects almost every positive-leaning case, not the top decile, so the")
print("    volume of actions is far higher than the policy intended")
print("  * expected-value arithmetic based on the stated probability is simply wrong")
print("  * you cannot compare scores across models or across time")
print("  * a threshold tuned last quarter means something different this quarter")
print("\nThe fix is one line (CalibratedClassifierCV) and it does not cost accuracy or AUC.")

**Exercise 4 (challenge).** You must ship a text classifier under hard constraints: 300
labelled examples, a 1 ms latency budget, and 50 MB of memory. Compare Naive Bayes against
logistic regression and a random forest on accuracy, latency and model size, and make a
recommendation.

In [ ]:
# --- Solution 4 -------------------------------------------------------------
import sys, pickle

SMALL_N = 300
Xs_tr, ys_tr = Xt_tr[:SMALL_N], yt_tr[:SMALL_N]
print(f"Training on {SMALL_N} labelled messages, evaluating on {len(yt_te)}\n")

contenders = {
    "MultinomialNB": make_pipeline(CountVectorizer(), MultinomialNB(alpha=0.5)),
    "BernoulliNB": make_pipeline(CountVectorizer(binary=True), BernoulliNB(alpha=0.5)),
    "logistic regression": make_pipeline(TfidfVectorizer(),
                                         LogisticRegression(max_iter=3000)),
    "linear SVM": make_pipeline(TfidfVectorizer(), LinearSVC(max_iter=5000, dual="auto")),
    "random forest": make_pipeline(CountVectorizer(),
                                   RandomForestClassifier(n_estimators=200, random_state=0)),
}
rows8 = []
for name, est in contenders.items():
    t0 = time.perf_counter(); est.fit(Xs_tr, ys_tr); fit_s = time.perf_counter() - t0
    cvv = cross_val_score(est, Xs_tr, ys_tr, cv=SKF).mean()
    t0 = time.perf_counter()
    for _ in range(3):
        est.predict(Xt_te[:200])
    lat_ms = (time.perf_counter() - t0) / (3 * 200) * 1000
    size_kb = len(pickle.dumps(est)) / 1024
    rows8.append({"model": name, "CV_acc_300": cvv, "test_acc": est.score(Xt_te, yt_te),
                  "fit_s": round(fit_s, 3), "latency_ms": round(lat_ms, 4),
                  "size_KB": round(size_kb, 1)})
study = pd.DataFrame(rows8).sort_values("test_acc", ascending=False)
print(study.to_string(index=False))
print(f"\nConstraints: latency < 1 ms, size < 51,200 KB")
study["meets_latency"] = study.latency_ms < 1.0
study["meets_size"] = study.size_KB < 51_200
print(study[["model", "test_acc", "latency_ms", "size_KB", "meets_latency",
             "meets_size"]].to_string(index=False))

In [ ]:
# How does each model behave as the label budget shrinks further?
fig, ax = plt.subplots(figsize=(8, 4.4))
budgets = [30, 60, 120, 300, 600, 1000]
for name, colour in [("MultinomialNB", "crimson"), ("logistic regression", "steelblue"),
                     ("random forest", "seagreen")]:
    accs = []
    for b in budgets:
        est = contenders[name]
        accs.append(cross_val_score(est, Xt_tr[:b], yt_tr[:b], cv=SKF).mean())
    ax.plot(budgets, accs, "o-", color=colour, label=name)
ax.set_xlabel("number of labelled examples"); ax.set_ylabel("CV accuracy")
ax.set_title("Behaviour under a label budget")
ax.legend(fontsize=8); plt.show()

print("=" * 72)
print("RECOMMENDATION")
print("=" * 72)
best = study.iloc[0]
nb_row = study[study.model == "MultinomialNB"].iloc[0]
print(f"Ship MultinomialNB.")
print(f"  accuracy  {nb_row.test_acc:.4f} vs the best model's {best.test_acc:.4f} "
      f"(difference {best.test_acc - nb_row.test_acc:+.4f})")
print(f"  latency   {nb_row.latency_ms:.4f} ms -- comfortably inside the 1 ms budget")
print(f"  size      {nb_row.size_KB:.1f} KB -- three orders of magnitude under the limit")
print(f"  fit time  {nb_row.fit_s:.3f} s, so retraining is instant")
print()
print("Reasoning against each alternative:")
print("  * the random forest is larger and slower for no accuracy gain at this label")
print("    budget, and its size grows with the vocabulary")
print("  * logistic regression is competitive and would be my choice if calibrated")
print("    probabilities were required -- but the brief asks for a classification, and")
print("    at 300 labels NB is at least as accurate (see the curve above)")
print("  * the linear SVM gives no probabilities at all without extra calibration cost")
print()
print("Caveats I would state to the team:")
print("  1. NB probabilities are over-confident. If the product later needs a confidence")
print("     score, wrap it in CalibratedClassifierCV -- cheap, and it does not cost accuracy.")
print("  2. Revisit the choice once we have a few thousand labels: the curve shows")
print("     logistic regression catching up and eventually passing NB.")
print("  3. Cap the vocabulary (min_df / max_features) so model size stays bounded as")
print("     new words arrive.")
print("  4. Monitor for vocabulary drift -- spam vocabulary changes deliberately and fast.")
print("=" * 72)

---
## Summary

| Concept | Key point |
|---|---|
| Rule | $\hat{y} = \arg\max_c P(c)\prod_j P(x_j\mid c)$ |
| Naive assumption | Features independent **given the class** — usually false |
| Why it works anyway | Only the argmax matters, and $O(p)$ parameters means low variance |
| Training | Counting. No optimisation, one pass |
| GaussianNB | Continuous features |
| MultinomialNB | Counts / TF-IDF — the text default |
| BernoulliNB | Binary presence/absence |
| ComplementNB | Imbalanced text |
| Laplace smoothing | `alpha`; prevents a single zero from vetoing a class |
| Log space | Add log-probabilities to avoid underflow |
| Calibration | Over-confident by construction; fix with `CalibratedClassifierCV` |
| Cannot learn | Interactions — it multiplies independent per-feature terms |
| Best use | Text, high-dimensional sparse data, tiny label budgets, hard latency limits |

**Next up:** [Notebook 9 — Introduction to Natural Language Processing](9.%20Introduction%20to%20NLP.ipynb),
where we take the text representation we used here and study it properly.